# CP3-06 修复错误：从失败检查点续跑

本节将 CP3-04 中 `node_joke` 的故障注入移除，然后使用同一个 `thread_id` 调用 `graph.invoke(None, config)`。目标不是从头重跑，而是让 LangGraph 从失败超步继续执行。

## 执行前提

必须先执行 `04_error.ipynb`，使 PostgreSQL 中存在 `chapter03-05` 的失败检查点。`05_find_error.ipynb` 只是读取历史，可选执行。若当前线程已经运行到 `END`，请清理该线程或换一个 `CP3_ERROR_THREAD_ID`。

本节的最小修复是去掉 `node_joke` 中的 `time.sleep(5)` 和 `raise RuntimeError(...)`；其余图结构保持不变。


In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, StateGraph
from loguru import logger

load_dotenv(override=True)
MODEL_NAME = os.getenv('DEEPSEEK_MODEL', 'deepseek-v4-flash')
DB_URL = os.getenv('LANGGRAPH_DB_URL')
if not os.getenv('DEEPSEEK_API_KEY'):
    raise RuntimeError('缺少 DEEPSEEK_API_KEY，请先配置 .env 或系统环境变量。')
if not DB_URL:
    raise RuntimeError('缺少 LANGGRAPH_DB_URL，请配置 PostgreSQL 连接串。')
model = ChatDeepSeek(model=MODEL_NAME, extra_body={'thinking': {'type': 'disabled'}})

class OverAllState(TypedDict, total=False):
    topic: str
    poem: str
    joke: str
    final_output: str

class InputState(TypedDict):
    topic: str

class OutputState(TypedDict):
    final_output: str

topics = ['布偶猫', '狸花猫', '金渐层']
topic_index = 1  # 恢复时不会执行 node_change_topic；该值只影响新起运行。

def node_change_topic(state: InputState) -> OverAllState:
    global topic_index
    sub_topic = topics[topic_index]
    topic_index = (topic_index + 1) % len(topics)
    return {'topic': f'{state["topic"]}:{sub_topic}'}

def node_poem(state: OverAllState) -> OverAllState:
    logger.info('node_poem 正在执行')
    response = model.invoke([HumanMessage(content=f'写一首关于{state["topic"]}主题的七言绝句')])
    return {'poem': response.content}

# 最小修复：删除故障注入，让上次失败的任务可以重新执行。
def node_joke(state: OverAllState) -> OverAllState:
    logger.info('node_joke 正在执行')
    response = model.invoke([HumanMessage(content=f'写一个关于{state["topic"]}主题的笑话')])
    return {'joke': response.content}

def node_output(state: OverAllState) -> OutputState:
    logger.info('node_output 正在执行')
    return {
        'final_output': (
            f'关于{state["topic"]}的七言绝句:{state["poem"]}'
            + chr(10)
            + f'笑话:{state["joke"]}'
        )
    }

builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)
builder.add_node('node_change_topic', node_change_topic)
builder.add_node('node_poem', node_poem)
builder.add_node('node_joke', node_joke)
builder.add_node('node_output', node_output)
builder.add_edge(START, 'node_change_topic')
builder.add_edge('node_change_topic', 'node_poem')
builder.add_edge('node_change_topic', 'node_joke')
builder.add_edge('node_poem', 'node_output')
builder.add_edge('node_joke', 'node_output')
builder.add_edge('node_output', END)

from langgraph.checkpoint.postgres import PostgresSaver
THREAD_ID = os.getenv('CP3_ERROR_THREAD_ID', 'chapter03-05')
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)
    config = {'configurable': {'thread_id': THREAD_ID}}
    latest = graph.get_state(config=config)
    if not latest.next:
        raise RuntimeError('当前线程没有待执行节点，请先执行 CP3-04 产生失败检查点，或更换 CP3_ERROR_THREAD_ID。')
    # get_state 会合并可回写的成功 pending write，因此这里通常只剩 node_joke。
    print({'resume_from': latest.next, 'saved_keys': sorted(latest.values.keys())})

    # None 表示使用 config 指向的检查点，不提交新的输入。
    result = graph.invoke(None, config=config)
    print(result)


## 预期结果与边界

- `node_change_topic` 不会再次执行；主题仍来自失败检查点。
- 已成功写出的 `poem` 可能通过 cached write 复用；失败且没有结果的 `node_joke` 会重新执行。
- 两个任务完成后，`node_output` 才会生成最终结果。
- 这不是保证文本完全一致的重放：重新调用 LLM 的节点仍可能返回不同内容。

如果代码提示没有待执行节点，通常是该线程已经恢复完成，而不是修复代码失效。清理线程或改用新的 `CP3_ERROR_THREAD_ID`，并重新按 04 → 05 → 06 顺序实验。
